<a href="https://colab.research.google.com/github/amirgroup-codes/ProtoMech/blob/main/ProtoMech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="left">
  <img src="https://raw.githubusercontent.com/amirgroup-codes/ProtoMech/main/ProtoMech_Logo_Glow.svg"
       alt="ProtoMech"
       width="60%">
</p>

# ProtoMech: Protein Circuit Tracing via Cross-layer Transcoders</h1>

ProtoMech is a framework for discovering computational circuits in protein language models using cross-layer transcoders. This colab notebook is designed to produce the four files required for our [website](https://protmech.github.io/):
1. `activation_indices.json`
2. `seq.txt`
3. `top_activations.json`
4. `virtual_weights.json`

A link to the paper can be found [here](https://arxiv.org/abs/2602.12026). We additionally provide our [code](https://github.com/amirgroup-codes/ProtoMech), [models](https://huggingface.co/ktalreja/ProtoMechModels), and [data](https://huggingface.co/datasets/ktalreja/ProtoMechData).

---

In [ ]:
# @title 0. Check GPU status and install dependencies
import torch
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Code may run extremely slowly.")
    print("----------------------------------------------------------------")
    print("TO FIX THIS:")
    print("1. Click 'Runtime' in the top menu.")
    print("2. Select 'Change runtime type'.")
    print("3. Under 'Hardware accelerator', select 'T4 GPU'.")
    print("4. Click 'Save' and then re-run this cell.")
    print("----------------------------------------------------------------")

import os
import sys
import pty
import shutil
import tarfile
import subprocess
import torch
import argparse
if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([argparse.Namespace])
from google.colab import files
from huggingface_hub import hf_hub_download
import ipywidgets as widgets
from IPython.display import display
!pip install -q huggingface_hub torch pytorch-lightning fair-esm

In [ ]:
# @title 1. Clone ProtoMech and download models and data
REPO_URL = "https://github.com/amirgroup-codes/ProtoMech.git"
BRANCH = "main"
TARGET_DIR = "/content/ProtoMech"

# @markdown Select the ESM2 model:
model_choice = "ESM2-35M" # @param ["ESM2-8M","ESM2-35M"]

# 1. Clean up or check for existing directory
if os.path.exists(TARGET_DIR):
    print(f"Directory {TARGET_DIR} already exists. Preparing for a fresh clone...")
    shutil.rmtree(TARGET_DIR)

# 2. Clone the repository (Public URL)
print(f"Cloning {REPO_URL}...")
result = os.system(f"git clone -q -b {BRANCH} {REPO_URL} {TARGET_DIR}")

if result == 0:
    print(f"Successfully cloned ProtoMech into {TARGET_DIR}")
else:
    print(f"ERROR: Failed to clone repository.")

# 3. Add to sys.path so the patches can find the files immediately
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)



# Download models
print('Downloading models...')
model_choice = globals().get("model_choice", "ESM2-8M")
MODELS_DIR = os.path.join(TARGET_DIR, "models")
VISUALIZATION_DIR = os.path.join(TARGET_DIR, "visualization")
MODEL_REPO = "ktalreja/ProtoMechModels"
DATA_REPO = "ktalreja/ProtoMechData"

if model_choice == "ESM2-8M":
    esm_filename = "esm2_t6_8M_UR50D.pt"
    clt_filename = "CLT_L6_D3200/checkpoints/last.ckpt"
    families_archive = "families.tar.gz"
    functions_archive = "functions.tar.gz"
    activations_filename = "top10_activations.pt"
    family_dir = os.path.join(TARGET_DIR, "family_circuit/families")
    function_dir = os.path.join(TARGET_DIR, "function_circuit/functions")
else:
    esm_filename = "esm2_t12_35M_UR50D.pt"
    clt_filename = "CLT_L12_D4800/last.ckpt"
    families_archive = "families_35M.tar.gz"
    functions_archive = "functions_35M.tar.gz"
    activations_filename = "top10_activations_35M.pt"
    family_dir = os.path.join(TARGET_DIR, "family_circuit/families_35M")
    function_dir = os.path.join(TARGET_DIR, "function_circuit/functions_35M")

esm_weights = os.path.join(MODELS_DIR, esm_filename)
clt_checkpoint = os.path.join(MODELS_DIR, clt_filename)
activations_pt = os.path.join(VISUALIZATION_DIR, activations_filename)

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VISUALIZATION_DIR, exist_ok=True)
os.makedirs(family_dir, exist_ok=True)
os.makedirs(function_dir, exist_ok=True)

hf_hub_download(
    repo_id=MODEL_REPO,
    filename=esm_filename,
    repo_type="model",
    local_dir=MODELS_DIR,
    local_dir_use_symlinks=False
)
hf_hub_download(
    repo_id=MODEL_REPO,
    filename=clt_filename,
    repo_type="model",
    local_dir=MODELS_DIR,
    local_dir_use_symlinks=False
)

# Download datasets
hf_hub_download(
    repo_id=DATA_REPO,
    filename=activations_filename,
    repo_type="dataset",
    local_dir=VISUALIZATION_DIR,
    local_dir_use_symlinks=False
)

def download_and_extract(filename, target_dir):
    print(f"Processing {filename}...")
    # 1. Download the tar.gz file
    tar_path = hf_hub_download(
        repo_id=DATA_REPO,
        filename=filename,
        repo_type="dataset",
        local_dir=TARGET_DIR # Download to root first
    )
    # 2. Extract into the specific target folder
    print(f"Extracting into {target_dir}...")
    try:
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=target_dir)
        print("Success.")
    except Exception as e:
        print(f"Error extracting {filename}: {e}")

download_and_extract(families_archive, family_dir)
download_and_extract(functions_archive, function_dir)

if not os.path.exists(esm_weights):
    raise ValueError(f"ESM not found at {esm_weights}, check download")
if not os.path.exists(clt_checkpoint):
    raise ValueError(f"CLT not found at {clt_checkpoint}, check download")
if not os.path.exists(activations_pt):
    raise ValueError(f"{activations_pt} not found, check download")

### 1.5 Discover your own circuit (optional)
Use this section to train a probe and discover a circuit for your own custom dataset. We follow a similar protocol to Appendix D of the paper.

### **Input CSV format**
Your CSV must have two specific columns (not case-sensitive):
1.  **Sequence**: `sequence` or `mutated_sequence`.
2.  **Score**: `score`, `DMS_score`, or `class`

#### **Option A: Binary classification**
* **Sequence:** can vary in length.
* **Score:** Must contain **only** `0` and `1`.
* **Example:**
    ```csv
    sequence,class
    MKV...AAA,1
    AGL...TTV,0
    MKV...AAB,1
    ```

#### **Option B: Regression**
* **Sequence:** must be same length.
* **Score:** Continuous numbers (float).
* **Example:**
    ```csv
    mutant,mutated_sequence,DMS_score
    K3R,MSR...LYK,3.74
    K3Q,MSQ...LYK,3.75
    K3E,MSE...LYK,3.67
    ```

In [ ]:
# @title Run circuit discovery
# @markdown **Settings**

# @markdown Select the type of task:
task_type = "Binary classification" # @param ["Binary classification", "Regression"]
# @markdown Check to upload your CSV file:
upload_csv = True # @param {type:"boolean"}
# @markdown Output folder name:
output_dir = "custom_circuit" # @param {type:"string"}
# @markdown Where to save the results:
external_path = "/content/experiments" # @param {type:"string"}

REPO_ROOT = "/content/ProtoMech"
SCRIPT_PATH = os.path.join(REPO_ROOT, "visualization", "auto_discover_circuit_website.py")

# --- 1. Input Handling ---
if upload_csv:
    print("Please upload your CSV file...")
    uploaded = files.upload()
    if not uploaded:
        sys.exit("Upload cancelled.")
    csv_name = list(uploaded.keys())[0]
    csv_path = os.path.join(os.getcwd(), csv_name)
else:
    sys.exit("Check 'upload_csv' to proceed.")
is_binary = (task_type == "Binary classification")
entry_name = os.path.splitext(csv_name)[0]
full_output_dir = os.path.join(external_path, output_dir)
DISCOVERY_OUTPUT_DIR = output_dir
DISCOVERY_FULL_OUTPUT_DIR = full_output_dir

# --- 2. Run Command ---
cmd = [
    "python", SCRIPT_PATH,
    "--csv_path", csv_path,
    "--is_binary", str(is_binary),
    "--output_dir", full_output_dir,
    "--entry_name", entry_name,
    "--clt_checkpoint", clt_checkpoint,
    "--esm_weights", esm_weights,
    "--model_size", model_choice,
    "--batch_size", "8"
]
print(f"\nStarting Discovery ({task_type})...")
print(f"   Input: {csv_name}")
print(f"   Output: {full_output_dir}/{entry_name}.json\n")
master, slave = pty.openpty()
p = subprocess.Popen(cmd, stdout=slave, stderr=slave, close_fds=True)
os.close(slave)
try:
    while True:
        try:
            data = os.read(master, 1024).decode()
            if not data: break
            sys.stdout.write(data)
            sys.stdout.flush()
        except OSError: break
except Exception: pass
p.wait()
os.close(master)

if p.returncode == 0:
    print(f"\nDone! Download your JSON here: {full_output_dir}/{entry_name}.json")
else:
    print("\nDiscovery Failed.")

In [ ]:
# @title 2. Add sequences to compute
# @markdown Example sequences:
# @markdown - Seq 1 (wildtype): `QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE`
# @markdown - Seq 2: `QYKLILNGKTLKGETTTEAVDAWTAEKVFKQYANDNGVDGEWTYDDATKTFTVTE`
# @markdown - Seq 3: `QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEQTYDDATKTFTVTE`

# @markdown Note: Use `Add Sequence +` to compare variations of the same protein (e.g., assessing the effect of mutations on a wildtype sequence). To discover a completely new circuit for a different protein, please reset and run the discovery pipeline again.

class SequenceInputManager:
    def __init__(self):
        self.sequences = []
        self.container = widgets.VBox()
        self.add_button = widgets.Button(description="Add Sequence +", icon="plus")
        self.add_button.on_click(self.add_field)

        # Initial field
        self.add_field(None)

    def add_field(self, b):
        idx = len(self.sequences) + 1
        label = "Seq 1 (wildtype):" if idx == 1 else f"Seq {idx}:"
        text = widgets.Text(placeholder=f"Enter protein sequence {idx}...", layout=widgets.Layout(width='80%'))
        box = widgets.HBox([widgets.Label(value=label, layout=widgets.Layout(width='120px')), text])
        self.sequences.append(text)
        self.container.children = tuple(list(self.container.children) + [box])

    def get_sequences(self):
        return [w.value.strip() for w in self.sequences if w.value.strip()]

    def display(self):
        display(widgets.VBox([self.container, self.add_button]))

# Instantiate and display
seq_manager = SequenceInputManager()
seq_manager.display()

In [ ]:
# @title 3. Generate Circuit Files (Run after adding sequences)
# @markdown **Configuration**
# @markdown - `circuit` (optional): Leave empty to auto-generate from `Seq 1`. You can also find a list of pre-discovered circuits [here](https://github.com/amirgroup-codes/ProtoMech/blob/main/visualization/circuits.md).
# @markdown - `upload_custom_circuit`: Check to upload a custom JSON.
# @markdown - `output_dir`: Name of the folder to create.

# --- 1. Get Inputs from Widget ---
sequences = seq_manager.get_sequences()
circuit = "SPG1_STRSG_Olson_2014" # @param {type:"string"}
upload_custom_circuit = False # @param {type:"boolean"}
output_dir = "GB1" # @param {type:"string"}
external_path = "/content/experiments" # @param {type:"string"}

# --- Validation ---
if not sequences:
    print("❌ Error: No sequences provided. Please add sequences in the UI above.")
    sys.exit(1)
print(f"Processing {len(sequences)} sequences...")
print(f"   Seq 1 (wldtype): {sequences[0][:10]}...")

# --- Setup Paths ---
REPO_ROOT = "/content/ProtoMech"
SCRIPTS_DIR = os.path.join(REPO_ROOT, "visualization")
FULL_OUTPUT_DIR = os.path.join(external_path, output_dir)
DISCOVERY_ENTRY_NAME = globals().get("entry_name", None)
DISCOVERY_FULL_OUTPUT_DIR = globals().get("DISCOVERY_FULL_OUTPUT_DIR", None)
if DISCOVERY_FULL_OUTPUT_DIR and os.path.abspath(DISCOVERY_FULL_OUTPUT_DIR) != os.path.abspath(FULL_OUTPUT_DIR):
    output_dirs = [FULL_OUTPUT_DIR, DISCOVERY_FULL_OUTPUT_DIR]
else:
    output_dirs = [FULL_OUTPUT_DIR]
existing_output_dir = FULL_OUTPUT_DIR
for candidate_dir in output_dirs:
    if DISCOVERY_ENTRY_NAME:
        probe_candidate = os.path.join(candidate_dir, f"{DISCOVERY_ENTRY_NAME}_probe.pt")
        circuit_candidate = os.path.join(candidate_dir, f"{DISCOVERY_ENTRY_NAME}_circuit.json")
    else:
        probe_candidate = os.path.join(candidate_dir, f"{output_dir}_probe.pt")
        circuit_candidate = os.path.join(candidate_dir, f"{output_dir}_circuit.json")
    if os.path.exists(probe_candidate) or os.path.exists(circuit_candidate):
        existing_output_dir = candidate_dir
        break
else:
    if len(output_dirs) > 1:
        existing_output_dir = output_dirs[1]
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)
if DISCOVERY_ENTRY_NAME:
    probe_path = os.path.join(existing_output_dir, f"{DISCOVERY_ENTRY_NAME}_probe.pt")
    existing_circuit_json = os.path.join(existing_output_dir, f"{DISCOVERY_ENTRY_NAME}_circuit.json")
else:
    probe_path = os.path.join(existing_output_dir, f"{output_dir}_probe.pt")
    existing_circuit_json = os.path.join(existing_output_dir, f"{output_dir}_circuit.json")

if not os.path.exists(SCRIPTS_DIR):
    raise FileNotFoundError(f"Could not find directory: {SCRIPTS_DIR}")
os.chdir(SCRIPTS_DIR)
def run_realtime(command):
    """
    Runs a command with a pseudo-terminal to allow real-time
    output and progress bars in Colab.
    """
    master, slave = pty.openpty()
    p = subprocess.Popen(command, stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    try:
        while True:
            try:
                data = os.read(master, 1024)
                if not data: break
            except OSError:
                break
    except Exception as e:
        pass #
    p.wait()
    os.close(master)
    return p.returncode, ""



circuit_json_path = None
needs_generation = False

# Helper: try a primary candidate path plus an alternate path with a repeated base folder prefix.
def resolve_candidate_path(base_dir, *parts):
    candidate = os.path.join(base_dir, *parts)
    if os.path.exists(candidate):
        return candidate
    if len(parts) >= 1:
        alt_part = os.path.join(os.path.basename(base_dir), parts[0])
        alt_candidate = os.path.join(base_dir, alt_part, *parts[1:])
        if os.path.exists(alt_candidate):
            return alt_candidate
    return None


def find_discovered_circuit_json():
    if DISCOVERY_FULL_OUTPUT_DIR and DISCOVERY_ENTRY_NAME:
        candidate = os.path.join(DISCOVERY_FULL_OUTPUT_DIR, f"{DISCOVERY_ENTRY_NAME}.json")
        if os.path.exists(candidate):
            return candidate
    return None


def find_any_circuit_json(search_dir):
    if not os.path.isdir(search_dir):
        return None
    candidates = sorted([f for f in os.listdir(search_dir) if f.endswith("_circuit.json")])
    for fn in candidates:
        path = os.path.join(search_dir, fn)
        if os.path.isfile(path):
            return path
    return None

discovered_json_path = find_discovered_circuit_json()
# CASE A: User wants to upload a file
if upload_custom_circuit:
    print("\nPlease upload custom circuit json...")
    uploaded = files.upload()
    if not uploaded:
        print("Upload cancelled. Aborting.")
        sys.exit(1)
    filename = list(uploaded.keys())[0]
    target_path = os.path.join(FULL_OUTPUT_DIR, filename)
    os.rename(filename, target_path)
    circuit_json_path = target_path

# CASE B: Prefer a discovery result from section 1.5 when available.
elif discovered_json_path:
    print(f"Using circuit JSON discovered in section 1.5: {discovered_json_path}")
    circuit_json_path = discovered_json_path

# CASE C: User specified a circuit query
elif circuit.strip():
    clean_query = circuit.strip()
    json_filename = clean_query if clean_query.endswith(".json") else f"{clean_query}.json"
    base_name = os.path.splitext(clean_query)[0]

    if clean_query.startswith("IPR"):
        candidate_path = resolve_candidate_path(family_dir, "CLT_sequential", json_filename)
        if candidate_path:
            circuit_json_path = candidate_path
        else:
            print(f"Warning: Could not find family circuit. Auto-generating...")
            needs_generation = True
    else:
        candidate_path = resolve_candidate_path(function_dir, "CLT_sequential", "multiples", base_name, "rand_multiples_fold0.json")
        if candidate_path:
            circuit_json_path = candidate_path
        else:
            print(f"Warning: Could not find function file. Auto-generating...")
            needs_generation = True

# CASE D: No input provided
else:
    if existing_circuit_json and os.path.exists(existing_circuit_json):
        print(f"Reusing circuit JSON from previous discovery output: {existing_circuit_json}")
        circuit_json_path = existing_circuit_json
    else:
        search_dirs = [FULL_OUTPUT_DIR]
        if DISCOVERY_FULL_OUTPUT_DIR and os.path.abspath(DISCOVERY_FULL_OUTPUT_DIR) != os.path.abspath(FULL_OUTPUT_DIR):
            search_dirs.insert(0, DISCOVERY_FULL_OUTPUT_DIR)
        for search_dir in search_dirs:
            candidate_path = find_any_circuit_json(search_dir)
            if candidate_path:
                print(f"Found circuit JSON in {search_dir}: {candidate_path}")
                circuit_json_path = candidate_path
                break
        if circuit_json_path is None:
            print("Auto-generating circuit from seq 1...")
            needs_generation = True



# --- Pipeline Execution ---
# Step 0: Generate Circuit (from Seq 1)
if needs_generation:
    print("\n[Step 0] Generating circuit JSON...")
    generated_json_path = os.path.join(FULL_OUTPUT_DIR, f"{output_dir}_circuit.json")
    if not os.path.exists(clt_checkpoint):
        raise FileNotFoundError(f"CLT checkpoint not found: {clt_checkpoint}")
    if not os.path.exists(esm_weights):
        raise FileNotFoundError(f"ESM model not found: {esm_weights}")
    cmd = [
        "python", "circuit_top_acts.py",
        "--sequence", sequences[0],
        "--output", generated_json_path,
        "--clt_checkpoint", clt_checkpoint,
        "--esm_path", esm_weights
    ]
    exit_code, cmd_output = run_realtime(cmd)
    if exit_code != 0:
        print("Error generating circuit")
        circuit_json_path = None
    else:
        circuit_json_path = generated_json_path

# Step 1: Analyze All Sequences
if circuit_json_path:
    print("\n[Step 1] Running multi-sequence analysis...")
    if os.path.exists(output_dir):
        if os.path.islink(output_dir):
            os.unlink(output_dir)
        elif os.path.isdir(output_dir):
            shutil.rmtree(output_dir)
    os.symlink(FULL_OUTPUT_DIR, output_dir)
    cmd_analysis = [
        "python", "circuit_analysis_builder_website.py",
        "--entry_name", output_dir,
        "--circuit_json", circuit_json_path,
        "--clt_ckpt", clt_checkpoint,
        "--esm_path", esm_weights,
        "--activations_pt", activations_pt,
        "--sequences"
    ] + sequences

    if os.path.exists(probe_path):
        cmd_analysis[4:4] = ["--probe_path", probe_path]
    else:
        print(f"No probe file found at {probe_path}; running analysis without probe.")
    exit_code, cmd_output = run_realtime(cmd_analysis)

    if exit_code != 0:
        print("Error conducting circuit analysis")
    else:
        print("\n[Step 2] Computing edge weights for each sequence (may take some time)...")

        subfolders = [f.path for f in os.scandir(FULL_OUTPUT_DIR) if f.is_dir() and "seq" in f.name]
        subfolders.sort() # Ensure seq1, seq2 order
        for folder_path in subfolders:
            folder_name = os.path.basename(folder_path)
            print(f"\n   Processing {folder_name}...")
            target_rel_path = os.path.join(output_dir, folder_name)
            cmd_weights = [
                "python", "get_edge_weights.py",
                "--base_folder", target_rel_path,
                "--clt_ckpt", clt_checkpoint,
                "--esm_path", esm_weights
            ]
            w_exit, w_out = run_realtime(cmd_weights)
            if w_exit != 0:
                print(f"Failed to compute weights for {folder_name}")

        print("\n==========")
        print(f"Results saved to: {FULL_OUTPUT_DIR}")
        print("==========")
        print("Files generated:")
        for f in os.listdir(FULL_OUTPUT_DIR):
             suffix = "/" if os.path.isdir(os.path.join(FULL_OUTPUT_DIR, f)) else ""
             print(f" - {f}{suffix}")

    # Cleanup Symlink
    if os.path.islink(output_dir):
        os.unlink(output_dir)
else:
    if not needs_generation:
        print("Aborted: No valid circuit JSON found.")